In [6]:
import pandas as pd
import numpy as np
import os

RAW_PATH = "../data_engine2/raw/sympscan-symptomps-to-disease/Diseases_and_Symptoms_dataset.csv" 

df_symptoms = pd.read_csv(RAW_PATH)

print(f"1. Raw Dataset loaded successfully! Shape: {df_symptoms.shape}")

ghost_cols = [col for col in df_symptoms.columns if "Unnamed" in col]
if ghost_cols:
    df_symptoms = df_symptoms.drop(columns=ghost_cols)

# Standardize column names (lowercase, replace spaces/hyphens with underscores)
df_symptoms.columns = (
    df_symptoms.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

target_col = "diseases"

symptom_features = [col for col in df_symptoms.columns if col != target_col]
print(f"2. Target diagnosis column identified as: '{target_col}'")
print(f"3. Isolated {len(symptom_features)} unique symptom features.")

# Force all symptom columns to be clean binary integers (0 or 1)
for col in symptom_features:
    df_symptoms[col] = pd.to_numeric(df_symptoms[col], errors="coerce").fillna(0).astype(int)

# Strip trailing whitespace from disease names
df_symptoms[target_col] = df_symptoms[target_col].astype(str).str.strip()

# Save the ML-ready matrix
os.makedirs("../data_engine2/processed", exist_ok=True)
processed_path = "../data_engine2/processed/symptom_triage_matrix.csv"
df_symptoms.to_csv(processed_path, index=False)

print(f"\nSUCCESS! Cleaned matrix saved to: {processed_path}")
print(f"Total Patient Records: {len(df_symptoms)}")
print(f"Total Unique Diseases: {df_symptoms[target_col].nunique()}")

1. Raw Dataset loaded successfully! Shape: (96088, 231)
2. Target diagnosis column identified as: 'diseases'
3. Isolated 230 unique symptom features.

SUCCESS! Cleaned matrix saved to: ../data_engine2/processed/symptom_triage_matrix.csv
Total Patient Records: 96088
Total Unique Diseases: 100


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, top_k_accuracy_score

df = pd.read_csv("../data_engine2/processed/symptom_triage_matrix.csv")
target_col = df.columns[0]

X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training on {len(X_train)} records | Testing on {len(X_test)} records...")

triage_model = RandomForestClassifier(
    n_estimators=150, 
    min_samples_split=5, 
    random_state=42, 
    n_jobs=-1
)
triage_model.fit(X_train, y_train)

# Evaluate Top-1 vs. Top-3 Clinical Accuracy
y_pred_top1 = triage_model.predict(X_test)
y_pred_proba = triage_model.predict_proba(X_test)

top1_acc = accuracy_score(y_test, y_pred_top1)
top3_acc = top_k_accuracy_score(y_test, y_pred_proba, k=3, labels=triage_model.classes_)

print("\n--- Diagnostic Model Results ---")
print(f"Top-1 Diagnostic Accuracy:  {top1_acc * 100:.2f}%")
print(f"Top-3 Clinical Accuracy:    {top3_acc * 100:.2f}%")

Training on 76870 records | Testing on 19218 records...

--- Diagnostic Model Results ---
Top-1 Diagnostic Accuracy:  87.37%
Top-3 Clinical Accuracy:    98.04% (Target is > 90%)


In [ ]:
import os
import joblib
import sklearn
from datetime import datetime

os.makedirs("../models", exist_ok=True)

engine2_bundle = {
    "model": triage_model,                         
    "feature_names": list(X.columns),              
    "classes": list(triage_model.classes_),      
    "metadata": {
        "engine_version": "1.0.0",
        "algorithm": "RandomForestClassifier",
        "training_records": len(X_train),
        "symptom_count": len(X.columns),
        "disease_count": len(triage_model.classes_),
        "top1_accuracy": float(top1_acc),
        "top3_accuracy": float(top3_acc),
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "sklearn_version": sklearn.__version__
    }
}

# Serialize to disk using joblib
artifact_path = "../models/engine2_triage_pipeline.joblib"
joblib.dump(engine2_bundle, artifact_path)
print(f"Clean Engine 2 bundle exported to: {artifact_path}")


Clean Engine 2 bundle exported to: ../models/engine2_triage_pipeline.joblib

--- Simulating API Reload & Inference ---


NameError: name 'np' is not defined

In [12]:
# PRODUCTION VERIFICATION TEST-
print("\n--- Simulating API Reload & Inference ---")
loaded_bundle = joblib.load(artifact_path)
loaded_model = loaded_bundle["model"]

# Create a mock patient
sample_patient = pd.DataFrame([[0]*len(loaded_bundle["feature_names"])], columns=loaded_bundle["feature_names"])
for symp in ["fever", "headache", "chills", "cough", "fatigue", "nausea"]:
    if symp in sample_patient.columns:
        sample_patient[symp] = 1

# Generate Top-3 Differential Diagnosis
probs = loaded_model.predict_proba(sample_patient)[0]
top3_indices = np.argsort(probs)[::-1][:3]

print("Top-3 Diagnostic Predictions for Patient:")
for rank, idx in enumerate(top3_indices, 1):
    disease = loaded_bundle["classes"][idx]
    confidence = probs[idx] * 100
    print(f"  {rank}. {disease} ({confidence:.1f}% confidence)")


--- Simulating API Reload & Inference ---


Top-3 Diagnostic Predictions for Patient:
  1. infectious gastroenteritis (25.8% confidence)
  2. common cold (20.7% confidence)
  3. strep throat (15.3% confidence)
